In [1]:
import pandas as pd
import numpy as np
import torch.nn as nn
import torch
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from torchvision import transforms
import os
import torch.nn.functional as F


pip = transforms.Compose([
    transforms.ToTensor()
])

class SofianeData(Dataset):
    def __init__(self, path, transforms, is_train):
        self.path = path
        self.images = [x.split('_')[1].split('.')[0] for x in os.listdir(os.path.join(self.path, 'images'))]
        self.transforms = transforms
        self.is_train = is_train
    def __len__(self):
        return len(self.images)
    def __getitem__(self, index):
        if self.is_train:
            return self.transforms(Image.open(os.path.join(self.path,'images',f'img_{self.images[index]}.png')).convert('RGB')),self.transforms(Image.open(os.path.join(self.path,'masks',f'mask_{self.images[index]}.png')).convert('L'))
        else:
            return self.transforms(Image.open(os.path.join(self.path,'images',f'img_{self.images[index]}.png')).convert('RGB'))

traind = SofianeData('train',pip,True)
testd = SofianeData('test',pip,False)

train = DataLoader(traind,4,True)
test = DataLoader(testd,1,False)

In [ ]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024))
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(1024, 512)
        
        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(512, 256)
        
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(256, 128)
        
        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv4 = DoubleConv(128, 64)
        
        # Final Output Convolution
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)      # Output: 64 x 512 x 512
        x2 = self.down1(x1)   # Output: 128 x 256 x 256
        x3 = self.down2(x2)   # Output: 256 x 128 x 128
        x4 = self.down3(x3)   # Output: 512 x 64 x 64
        x5 = self.down4(x4)   # Output: 1024 x 32 x 32
        
        x = self.up1(x5)      # Output: 512 x 64 x 64
        x = torch.cat([x4, x], dim=1) # Output: 1024 x 64 x 64
        x = self.conv1(x)     # Output: 512 x 64 x 64
        
        x = self.up2(x)       # Output: 256 x 128 x 128
        x = torch.cat([x3, x], dim=1) # Output: 512 x 128 x 128
        x = self.conv2(x)     # Output: 256 x 128 x 128
        
        x = self.up3(x)       # Output: 128 x 256 x 256
        x = torch.cat([x2, x], dim=1) # Output: 256 x 256 x 256
        x = self.conv3(x)     # Output: 128 x 256 x 256
        
        x = self.up4(x)       # Output: 64 x 512 x 512
        x = torch.cat([x1, x], dim=1) # Output: 128 x 512 x 512
        x = self.conv4(x)     # Output: 64 x 512 x 512
        logits = self.outc(x) # Output: 1 x 512 x 512
        return logits


model = UNet(3,1)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr = 5e-4)

In [3]:
from tqdm.notebook import tqdm

def train_one_epoch():
    model.train()
    running_loss = 0.0
    pbar = tqdm(train)
    
    for images, masks in pbar:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        pbar.set_postfix(loss=loss.item())
        
    epoch_loss = running_loss / len(train.dataset)
    return epoch_loss

In [ ]:
for i in range(0, 14):
    print(f'EPoch is {i}')
    loss = train_one_epoch()
    print(f'loss is {loss}')